# Efektywne programowanie w języku Python
## Django — część 1
Od projektu do aplikacji notatek

### Cel i zakres
Opcjonalny blok kursu. Zbudujemy projekt z SQLite, modelem `Note`, panelem administratora oraz listą i szczegółami notatki.

Python 3.10 + Django **5.2 LTS**. Używamy dokumentacji 5.2; nie kopiujemy konfiguracji z Django 1.11.
Formularze użytkownika i dalszy rozwój aplikacji są tematem kolejnej części.

### Po tym wykładzie potrafisz
- Wyjaśnić drogę żądania od adresu URL do odpowiedzi.
- Odróżnić projekt od aplikacji i model od migracji.
- Zarejestrować model w adminie.
- Połączyć `ListView`, `DetailView`, routing i szablony.
- Sprawdzić działanie testem zamiast zgadywać na podstawie wyglądu strony.

## Czym jest Django?
Framework webowy Pythona łączący routing, ORM, szablony, uwierzytelnianie, formularze i panel administracyjny.

Powstał z potrzeb zespołu tworzącego serwisy prasowe; publicznie udostępniono go w 2005 roku. Nazwa nawiązuje do gitarzysty Django Reinhardta.

Najważniejsze dla nas: spójne narzędzia do budowy aplikacji, nie lista firm ani historyczne rekordy ruchu.

### Żądanie → odpowiedź
**Przeglądarka → middleware → URLconf → widok → odpowiedź HTTP**

Widok może pobrać dane z modelu i wyrenderować szablon HTML.
Middleware otacza obsługę żądania i odpowiedzi; może też zakończyć żądanie wcześniej.

Model opisuje dane, szablon prezentację, a widok przygotowuje odpowiedź — ten podział nazywamy często MTV.

### Projekt a aplikacja
**Projekt** zawiera konfigurację wdrożenia: ustawienia, główne URL-e, punkty wejścia WSGI/ASGI.

**Aplikacja** skupia funkcjonalność, np. `notes`: modele, widoki, szablony, testy.
Projekt używa wielu aplikacji. Aplikacja może być wielokrotnego użytku, jeśli nie zwiążemy jej nadmiernie z jednym projektem.

## Środowisko przed zajęciami
W aktywnym środowisku kursu, z katalogu repozytorium:
```text
python -m pip install -r part10_django_files/requirements.txt
python -m django --version
```
[Zależności](part10_django_files/requirements.txt) wybierają linię 5.2 i bieżące wydanie poprawkowe.
Nie instalujemy pakietów podczas „Run All”. Django 5.2 obsługuje Python 3.10; nowsze linie mogą wymagać nowszego Pythona.

In [1]:
import sys
import django
print("Python:", sys.version.split()[0])
print("Django:", django.get_version())
assert django.VERSION[:2] == (5, 2), "Ten wykład przygotowano dla Django 5.2"

Python: 3.10.4
Django: 5.2.19


### Dwie ścieżki demonstracji
1. **Budowa od zera:** wykonuj polecenia i zapisuj fragmenty w wskazanych plikach.
2. **Gotowy projekt:** [part10_django_files/demo](part10_django_files/demo/) zawiera kompletne pliki.

Fragmenty plików pokazujemy jako kod na slajdach, a nie komórki do uruchamiania w kernelu. Django działa jako osobny program.
[Instrukcja uruchomienia i testowania](part10_django_files/README.md).

### Nowy projekt od zera
W osobnym katalogu roboczym:
```text
mkdir django_notes
cd django_notes
python -m django startproject config .
python manage.py startapp notes
```
Kropka oznacza katalog docelowy projektu. Nazwa `config` jest konsekwentnie używana w przykładach.
Nie uruchamiaj `startproject` wewnątrz gotowej demonstracji.

### Mapa plików
```text
manage.py
config/          settings.py, urls.py, wsgi.py, asgi.py
notes/           apps.py, models.py, views.py, urls.py
                 admin.py, tests.py
  migrations/    0001_initial.py
  templates/notes/
                 base.html
                 note_list.html
                 note_detail.html
```
`manage.py` wykonuje polecenia z ustawieniami projektu. WSGI i ASGI są punktami wejścia dla serwerów aplikacyjnych.

## Ustawienia bez przepisywania całego pliku
Zachowaj ustawienia wygenerowane przez `startproject`. Zmieniaj tylko potrzebne fragmenty.

W nowym projekcie `BASE_DIR` korzysta z `pathlib.Path`; nie wymaga `__file__` w komórce notebooka.
`__file__` oznacza plik modułu, a nie zawsze istniejący element sesji interaktywnej.

### Lokalna konfiguracja
Fragmenty `config/settings.py`:
```python
LANGUAGE_CODE = "pl"
TIME_ZONE = "Europe/Warsaw"
USE_I18N = True
USE_TZ = True
STATIC_URL = "static/"
DEFAULT_AUTO_FIELD = "django.db.models.BigAutoField"
```
`USE_L10N` usunięto z nowego Django. Przy `USE_TZ=True` używaj świadomych stref czasowych, np. `django.utils.timezone.now()`.

### `SECRET_KEY`: podpis, nie szyfrowanie
Klucz służy m.in. do kryptograficznego podpisywania danych i tokenów. Podpis pozwala wykryć zmianę danych, ale ich nie ukrywa.

Nie szyfruje JSON-a z hasłem i nie służy do przechowywania haseł użytkowników — Django korzysta z funkcji haszujących hasła.
Klucz produkcyjny pobieraj z konfiguracji środowiska, poza repozytorium. Gotowa demonstracja ma jawnie oznaczony klucz lokalny.

### `DEBUG` i `ALLOWED_HOSTS`
```python
DEBUG = True  # tylko lokalny rozwój
ALLOWED_HOSTS = ["127.0.0.1", "localhost"]
```
`ALLOWED_HOSTS` ogranicza dozwolone wartości nagłówka HTTP **Host**, nie adresy IP klientów.
Nie jest firewallem, mechanizmem logowania ani konfiguracją CORS.
Na produkcji potrzebne są rzeczywiste nazwy hostów i `DEBUG=False`.

### SQLite w demonstracji
```python
DATABASES = {"default": {
    "ENGINE": "django.db.backends.sqlite3",
    "NAME": BASE_DIR / "db.sqlite3",
}}
```
SQLite nie wymaga osobnego serwera. Django obsługuje też inne bazy, np. PostgreSQL.
Wybór dla wdrożenia zależy od wymagań, współbieżności i operacji, nie tylko prostoty instalacji.

### `INSTALLED_APPS` i middleware
Do wygenerowanej listy `INSTALLED_APPS` dopisz:
```python
"notes.apps.NotesConfig",
```
Zachowaj standardowe aplikacje i middleware. Ich kolejność ma znaczenie: uwierzytelnianie korzysta m.in. z sesji.
`startapp` tworzy pliki; samo polecenie nie instaluje aplikacji w projekcie.

### Szablony aplikacji
Domyślne ustawienie:
```python
"APP_DIRS": True,
```
pozwala szukać szablonów w katalogach `templates` zainstalowanych aplikacji.

Przestrzeń nazw katalogu `templates/notes/` ogranicza kolizje nazw. Dla katalogu szablonów projektu można dodatkowo ustawić `DIRS: [BASE_DIR / "templates"]`.

## Model notatki
Model definiuje pola i zachowanie danych, a ORM tworzy zapytania do bazy.

Zaczynamy od tytułu, treści i czasu utworzenia. Wszystkie notatki demonstracyjne będą publiczne — nie ma tu jeszcze podziału na właścicieli.

### notes/models.py
Plik: `notes/models.py`

```python
from django.db import models

class Note(models.Model):
    title = models.CharField("tytuł", max_length=256)
    body = models.TextField("treść")
    created_at = models.DateTimeField("utworzono", auto_now_add=True)

    class Meta:
        ordering = ["-created_at", "-pk"]
        verbose_name = "notatka"
        verbose_name_plural = "notatki"

    def __str__(self):
        return self.title
```

### Co wynika z modelu?
- `max_length=256` opisuje limit pola tytułu.
- `auto_now_add=True` ustawia czas utworzenia przy pierwszym zapisie.
- `__str__` daje czytelny opis, np. w panelu admina.
- `ordering` ustala stabilną kolejność; `-pk` rozstrzyga remisy czasu.

Walidacja formularza i ograniczenia bazy nie są tym samym. `save()` nie wywołuje automatycznie pełnej walidacji modelu.

### Migracje: plan zmiany a wykonanie
```text
python manage.py makemigrations notes
python manage.py migrate
python manage.py showmigrations notes
python manage.py sqlmigrate notes 0001
```
`makemigrations` zapisuje opis zmian schematu, a `migrate` go wykonuje. Migracje zapisujemy w repozytorium.
`migrate` nie wypełnia automatycznie tabel przykładowymi notatkami.

### Jak czytać migrację?
`0001_initial.py` zawiera `CreateModel` z polami `id`, `title`, `body`, `created_at`.

Pliku migracji zwykle nie przepisujemy ze slajdu — generujemy go z modelu. W gotowej demonstracji migracja już istnieje.

Kontrola zgodności:
```text
python manage.py makemigrations --check --dry-run
```

## Panel administratora
Admin to interfejs dla zaufanych użytkowników zarządzających danymi.
Jest dobrym narzędziem zaplecza, ale nie zastępuje dowolnego interfejsu użytkownika aplikacji.

Najpierw migracje, potem konto administratora i rejestracja modelu.

### notes/admin.py
Plik: `notes/admin.py`

```python
from django.contrib import admin
from .models import Note

@admin.register(Note)
class NoteAdmin(admin.ModelAdmin):
    list_display = ("title", "created_at")
    search_fields = ("title", "body")
    list_filter = ("created_at",)
    readonly_fields = ("created_at",)
```

### Konto i uruchomienie
W katalogu z `manage.py`:
```text
python manage.py migrate
python manage.py createsuperuser
python manage.py runserver 127.0.0.1:8000
```
Otwórz `/admin/`, zaloguj się i utwórz kilka notatek.
Hasło wpisuj interaktywnie; nie umieszczaj go w slajdach ani historii poleceń.
`runserver` służy do rozwoju, nie do produkcji.

### Demonstracja na żywo: admin
1. Otwórz listę notatek.
2. Dodaj tytuł i treść z polskimi znakami.
3. Spróbuj zapisać pusty tytuł — obejrzyj komunikat walidacji.
4. Wyszukaj fragment tytułu.

Nazwy pól, `list_display`, `search_fields` i `list_filter` wynikają z kodu, a nie ręcznie zbudowanych ekranów.

## Routing: `path`, `include`, nazwy
Adres projektu kieruje ruch do aplikacji, a aplikacja definiuje swoje ścieżki.

Korzystamy z `django.urls.path`. Dawne `django.conf.urls.url` usunięto.
Nazwane URL-e pozwalają generować odnośniki bez wpisywania ścieżek w wielu miejscach.

### config/urls.py
Plik: `config/urls.py`

```python
from django.contrib import admin
from django.urls import include, path
from django.views.generic import RedirectView

urlpatterns = [
    path("", RedirectView.as_view(pattern_name="notes:note-list", permanent=False)),
    path("admin/", admin.site.urls),
    path("notes/", include("notes.urls")),
]
```

### notes/urls.py
Plik: `notes/urls.py`

```python
from django.urls import path
from .views import NoteDetailView, NoteListView

app_name = "notes"
urlpatterns = [
    path("", NoteListView.as_view(), name="note-list"),
    path("<int:pk>/", NoteDetailView.as_view(), name="note-detail"),
]
```

### Namespace i konwerter
`app_name = "notes"` tworzy przestrzeń nazw: `notes:note-list`, `notes:note-detail`.

`<int:pk>` przekazuje widokowi liczbę. `/notes/abc/` nie pasuje do tej ścieżki.
Dla istniejącej ścieżki, ale nieistniejącego rekordu, `DetailView` zwraca **404**.
Namespace nie stanowi kontroli dostępu.

## Widoki generyczne
`ListView` pobiera kolekcję, `DetailView` jeden obiekt.
`as_view()` tworzy callable obsługujące żądanie dla klasy widoku.

Importujemy obie klasy; widok szczegółowy wymaga także osobnego szablonu.

### notes/views.py
Plik: `notes/views.py`

```python
from django.views.generic import DetailView, ListView
from .models import Note

class NoteListView(ListView):
    model = Note
    context_object_name = "notes"
    paginate_by = 5

class NoteDetailView(DetailView):
    model = Note
    context_object_name = "note"
```

### Domyślne nazwy szablonów i kontekstu
| Widok | Szablon | Nasza nazwa kontekstu |
|---|---|---|
| `NoteListView` | `notes/note_list.html` | `notes` |
| `NoteDetailView` | `notes/note_detail.html` | `note` |

Domyślnie dostępne są też `object_list` dla listy i `object` dla szczegółu. Nazwy ustawiamy jawnie dla czytelności.
`paginate_by=5` ogranicza liczbę notatek na stronie.

### Szablon bazowy
`notes/templates/notes/base.html` zawiera wspólny dokument HTML:
```html
<!doctype html>
<html lang="pl">
<head><meta charset="utf-8"><title>Notatki</title></head>
<body>
  <nav><a href="{% url 'notes:note-list' %}">Notatki</a></nav>
  <main>{% block content %}{% endblock %}</main>
</body>
</html>
```
Pełny plik demonstracji dodaje styl, responsywny viewport i blok tytułu.

### Lista notatek
Plik: `notes/templates/notes/note_list.html`

```html
{% extends "notes/base.html" %}
{% block content %}
<h1>Notatki</h1>
<ul>
{% for note in notes %}
  <li><a href="{% url 'notes:note-detail' note.pk %}">{{ note.title }}</a></li>
{% empty %}
  <li>Nie ma jeszcze notatek.</li>
{% endfor %}
</ul>
{% if is_paginated %}
```

### Paginacja listy
Plik: `notes/templates/notes/note_list.html`

```html
<p class="pagination">
  {% if page_obj.has_previous %}<a href="?page={{ page_obj.previous_page_number }}">Poprzednia</a>{% endif %}
  Strona {{ page_obj.number }} z {{ paginator.num_pages }}
  {% if page_obj.has_next %}<a href="?page={{ page_obj.next_page_number }}">Następna</a>{% endif %}
</p>
{% endif %}
{% endblock %}
```

### Szczegóły notatki
Plik: `notes/templates/notes/note_detail.html`

```html
{% extends "notes/base.html" %}
{% block title %}{{ note.title }} — Notatki{% endblock %}
{% block content %}
<h1>{{ note.title }}</h1>
<time>{{ note.created_at|date:"d.m.Y H:i" }}</time>
<article>{{ note.body }}</article>
<p><a href="{% url 'notes:note-list' %}">Wróć do listy</a></p>
{% endblock %}
```

### Autoescaping i dane użytkownika
`{{ note.body }}` domyślnie koduje znaki specjalne HTML. Tekst `<script>` ma być tekstem, nie uruchomionym skryptem.

Nie dodawaj `|safe` do niesprawdzonej treści użytkownika.
Autoescaping nie zastępuje autoryzacji ani ochrony wszystkich kontekstów, np. dowolnego kodu JavaScript.

### Gdzie kończy się ochrona admina?
Admin wymaga uprawnień, ale `/notes/` i `/notes/<pk>/` w tym przykładzie są publiczne.

W aplikacji prywatnych notatek potrzebne są logowanie oraz filtrowanie rekordów według właściciela również w szczegółach. Samo ukrycie linku nie chroni obiektu.
To naturalne rozwinięcie kolejnej części wykładu.

### ORM: utwórz i odczytaj dane
Uruchom:
```text
python manage.py shell
```
Następnie w powłoce Django:
```python
from notes.models import Note
note = Note.objects.create(title="Pierwsza", body="Treść")
Note.objects.filter(title__icontains="pierwsza")
Note.objects.get(pk=note.pk)
```
To polecenia zmieniające lokalną bazę demonstracyjną, nie komórki tego notebooka.

### Rozszerzenie: leniwe QuerySet
Budowanie `filter()` nie oznacza od razu pobrania wszystkich rekordów. Zapytanie jest wykonywane przy ewaluacji wyniku, np. iteracji.

`get()` oczekuje jednego rekordu, może zgłosić `DoesNotExist` albo `MultipleObjectsReturned`.
ORM pomaga parametryzować wartości; nie sklejaj SQL-a z tekstem użytkownika.

## Testy jako część demonstracji
Sprawdzamy zachowanie HTTP, szablony, pustą listę, brak rekordu, paginację, kodowanie HTML i dostęp do admina.

Testy używają oddzielnej bazy testowej. Zwykłe żądanie HTTP w przeglądarce korzysta z bazy projektu.

### Przykład testu pustej listy
Plik: `notes/tests.py`

```python
from django.contrib.auth import get_user_model
from django.test import TestCase
from django.urls import reverse
from .models import Note

class NoteViewsTests(TestCase):
    def test_empty_list(self):
        response = self.client.get(reverse("notes:note-list"))
        self.assertContains(response, "Nie ma jeszcze notatek.")
        self.assertTemplateUsed(response, "notes/note_list.html")

    def test_list_detail_and_escape(self):
```

### Uruchamianie testów projektu
W katalogu `demo`:
```text
python manage.py check
python manage.py test notes
```
Wygodna kontrola z katalogu repozytorium:
```text
python part10_django_files/verify_demo.py
```
Weryfikator kopiuje projekt do katalogu tymczasowego i sprawdza migracje oraz 8 testów. Nie zmienia roboczej bazy.

In [ ]:
from pathlib import Path
import os
import subprocess

verifier = Path("part10_django_files/verify_demo.py").resolve()
completed = subprocess.run(
    [sys.executable, str(verifier)], check=True,
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=120)
print(completed.stdout)

### Typowe problemy na zajęciach
| Objaw | Co sprawdzić? |
|---|---|
| `No module named django` | Interpreter i aktywne środowisko |
| `no such table` | `INSTALLED_APPS`, migracje |
| `TemplateDoesNotExist` | Ścieżkę i `APP_DIRS` |
| 404 | URL, namespace, identyfikator rekordu |
| Brak modelu w adminie | Rejestrację i uprawnienia |

Czytaj traceback od konkretnego wyjątku, zamiast zmieniać wiele plików naraz.

### Rozszerzenie: formularze i metody HTTP
GET powinien odczytywać dane. Zmiany stanu wykonujemy przez odpowiednie metody, zwykle POST w formularzu.

Formularze POST w szablonach Django wymagają `{% csrf_token %}` i włączonego middleware CSRF. Nie wyłączamy go, aby „naprawić” błąd 403.
W tym wykładzie własne widoki wyświetlają dane; edycję realizuje admin.

### Rozszerzenie: przed wdrożeniem
- `DEBUG=False`, prywatny `SECRET_KEY`, właściwe hosty.
- HTTPS, właściwa obsługa plików statycznych i serwer WSGI/ASGI.
- Aktualizacje, kopie bazy i kontrola uprawnień.
- `python manage.py check --deploy` dla konfiguracji docelowej.

Te punkty nie tworzą kompletnego wdrożenia. Nie wystawiamy konfiguracji demonstracyjnej do Internetu.

### Eksperymenty: zapytania i walidacja
Trzy opcjonalne demonstracje wykonujemy na osobnej bazie SQLite w pamięci.
Mini-modele `Author` i `Entry` mają relację właściciela, aby pokazać zapytania bez zmiany modelu `Note` z części pierwszej.

[Pełny skrypt](part10_django_files/experiments.py). Nie wykonuje migracji ani zapisów w bazie zajęć.

### N+1: przewidź liczbę zapytań
```python
[entry.owner.name for entry in Entry.objects.all()]
```
Dla trzech rekordów: jedno zapytanie o listę i trzy o autorów, nawet jeśli autor jest ten sam.
```python
[entry.owner.name for entry in
 Entry.objects.select_related("owner")]
```
`select_related` pobiera relację ForeignKey w zapytaniu JOIN. Dla relacji kolekcji rozważ `prefetch_related`, nie identyczny mechanizm.

### Kiedy QuerySet pyta bazę?
```python
queryset = Entry.objects.filter(owner=alice)  # jeszcze bez SQL
queryset.exists()    # pytanie o istnienie
queryset.count()     # zliczanie bez pobierania pełnej listy
list(queryset)      # pobranie rekordów
list(queryset)      # cache tego samego QuerySet
```
Po pełnej ewaluacji `count()` i `exists()` tego QuerySet mogą użyć cache. Nowy QuerySet lub jego klon nie musi mieć tych wyników.

### Formularz, model i baza: trzy warstwy
`ModelForm` z wymaganym tytułem odrzuca pusty tekst.
`objects.create(title="", ...)` nie wywołuje automatycznie `full_clean()` i w tej bazie zapisze pusty napis.

`blank=False` to reguła walidacji, a `NOT NULL` nie oznacza „niepusty napis”. Jeśli reguła musi obowiązywać dla każdego zapisu, rozważ również jawne ograniczenie bazy. Różne bazy mogą inaczej traktować puste napisy.

In [ ]:
experiment = Path("part10_django_files/experiments.py").resolve()
result = subprocess.run(
    [sys.executable, str(experiment), "--part", "1"], check=True,
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=60)
print(result.stdout)

### Jak czytać wynik eksperymentu?
`CaptureQueriesContext` liczy zapytania tylko wewnątrz zaznaczonego fragmentu.
Dla naszego wejścia otrzymujemy 4 zapytania zamiast 1 oraz jawnie sprawdzone etapy ewaluacji QuerySet.

Nie przenoś liczby zapytań na dowolny szablon: paginacja, middleware i dodatkowe relacje mogą zmienić cały koszt żądania.

## Ćwiczenie na zakończenie
1. Utwórz w adminie sześć notatek — sprawdź drugą stronę listy.
2. Otwórz szczegóły i nieistniejący identyfikator.
3. Wpisz `<b>tekst</b>` w treści — czy tekst staje się HTML-em?
4. Zmień `paginate_by` i przewidź liczbę stron.
5. Dodaj test nowego wymagania przed zmianą kodu.

### Pytania kontrolne
- Czym różni się `makemigrations` od `migrate`?
- Czy `SECRET_KEY` szyfruje treść notatki?
- Czy `ALLOWED_HOSTS` ogranicza adresy klientów?
- Dlaczego potrzebujemy `as_view()`?
- Czy panel admina automatycznie chroni publiczny `DetailView`?

### Odpowiedzi
- Opis zmiany schematu i wykonanie migracji to dwa etapy.
- Nie: podpisywanie nie zapewnia poufności.
- Nie: ustawienie dotyczy nagłówka Host.
- Klasa widoku musi zostać przekształcona w callable obsługujące żądanie.
- Nie: autoryzację trzeba zaprojektować dla każdego widoku i obiektu.

### Self reading — projekt i dane
- [Oficjalny tutorial Django 5.2](https://docs.djangoproject.com/en/5.2/intro/tutorial01/)
- [Modele i admin — tutorial 2](https://docs.djangoproject.com/en/5.2/intro/tutorial02/)
- [URL dispatcher](https://docs.djangoproject.com/en/5.2/topics/http/urls/)
- [Migracje](https://docs.djangoproject.com/en/5.2/topics/migrations/)
- [Zgodność wersji Pythona](https://docs.djangoproject.com/en/5.2/faq/install/)

Materiały dodatkowe są opcjonalne, zgodnie ze statusem całego bloku Django.

### Self reading — widoki i konfiguracja
- [Generyczne widoki wyświetlania](https://docs.djangoproject.com/en/5.2/topics/class-based-views/generic-display/)
- [Szablony i autoescaping](https://docs.djangoproject.com/en/5.2/topics/templates/)
- [Testowanie](https://docs.djangoproject.com/en/5.2/topics/testing/overview/)
- [Ustawienia](https://docs.djangoproject.com/en/5.2/ref/settings/)
- [Lista kontrolna wdrożenia](https://docs.djangoproject.com/en/5.2/howto/deployment/checklist/)

Stare materiały dla 1.11/2.2 nie są instrukcją do tego projektu.

### Dalsze rozszerzenia — po podstawach
Formularze, właściciel notatki, uprawnienia obiektowe, statyczne zasoby i relacje modeli.

Pakiety zewnętrzne, np. import/export czy rozbudowane logowanie, dobieramy do problemu po ocenie dokumentacji i zgodności wersji.
Nie wymagamy prezentacji przypisanych dawniej do konkretnych osób.